# FAN-Autoformer — Multi-Seed × Multi-Horizon Minimum Temperature Forecasting

**Architecture:** FAN-Autoformer (Frequency Adaptive Normalization + Autoformer with AutoCorrelation + Dynamic Decomposition Kernel)

**Task:** Daily minimum temperature (`tmin`) prediction at Ilam Airport, Iran  
**Horizons:** H ∈ {1, 3, 7, 30} days  
**Seeds:** 42, 123, 456, 789, 1011 (multi-seed reproducibility evaluation)  
**Dataset:** ~20 years of daily meteorological observations (Ilam Airport, Iran)

---

### Key Architectural Components

| Component | Role |
|---|---|
| **FANLayer** | Frequency Adaptive Normalization — extracts the dominant non-stationary (trend/seasonality) component via FFT, normalizes the stationary residual, and restores the original temperature scale at inference via an MLP-based denormalization head. Outperforms RevIN on strongly seasonal meteorological data. |
| **SimpleDecompositionBlock** | Moving-average trend/seasonal decomposition with an adaptively chosen odd kernel size based on `pred_len`. Ensures correct boundary behavior and prevents even-length padding artefacts. |
| **AutoCorrelation** | Replaces O(L²) dot-product attention with O(L log L) FFT cross-correlation. Discovers top-k time-delay lags and aggregates value vectors by circular shifting — ideal for periodic meteorological signals. |
| **EncoderLayer** | AutoCorrelation → residual + LayerNorm → decomposition × 2 + FFN. Returns seasonal component only. |
| **DecoderLayer** | Self-AutoCorrelation → cross-AutoCorrelation (with encoder output) → FFN → 3-stage decomposition. Returns seasonal + accumulated trend. |
| **LayerWiseInterpreter** | Registers forward/backward hooks on every attention and FFN sub-module; computes gradient × activation importance maps for XAI analysis after each epoch. |
| **AdvancedRealtimeVisualizer** | Produces four Plotly HTML dashboards per checkpoint: training progress, attention heatmap, layer-wise feature importance bars, and temporal importance trend lines. |


In [ ]:
# =============================================================================
# FAN-Autoformer — Imports and Global Setup
# =============================================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
import os, time
import gc       # explicit garbage collection for GPU/CPU memory management
import random

import kaleido  # required by Plotly for static image export (PNG/SVG)
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — safe for servers and notebooks
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings('ignore')


## Section 1 — Data Loading and Feature Engineering

The raw CSV (`ilam_weather.csv`) contains ~20 years of daily meteorological
observations from Ilam Airport, Iran.  We construct a rich causal feature set
that exposes multi-scale temporal patterns to FAN-Autoformer without any data leakage.
Every engineered feature uses only past observations relative to the prediction target
(`.shift(1)` or larger).

**Feature groups:**

| Group | Features | Purpose |
|---|---|---|
| Lag | tmin_lag{1,2,3,7,14,30} | Short- and medium-range autoregressive memory |
| Rolling stats | rolling_{mean,std,min,max}_{7,14,30} | Multi-scale trend and variability |
| EWM | ewm_{7,30} | Recency-weighted smoothed baseline |
| Difference | diff_{1,7} | Day-to-day and weekly change rate |
| Interaction | temp_range, humidity_range, pressure_range | Cross-variable meteorological signals |
| Cyclical time | month/day_of_year/day_of_week encoded as sin+cos | Continuous circular calendar structure |

> **Leakage prevention:** all meteorological covariates (tmax, p0max, p0min, sshn, umin, umax)
> are shifted by 1 day so the model never sees same-day measurements for the target `tmin`.


In [ ]:
# =============================================================================
# FAN-Autoformer — Data Loading
# =============================================================================
df = pd.read_csv('ilam_weather.csv', index_col=0, parse_dates=True)


def create_advanced_features(dataframe, col_target='tmin'):
    """Build the full engineered feature set for FAN-Autoformer.

    All features are strictly causal: no same-day information leaks into the
    encoder window.  The function applies .shift(1) to every covariate.

    Feature groups
    --------------
    1. **Lag features** (col_target_lag{1,2,3,7,14,30}):
       Raw historical values of the target at fixed look-back offsets.
       Lag 1–3: short-term persistence (nocturnal radiative cooling, synoptic advection).
       Lag 7: weekly meteorological cycle.
       Lags 14, 30: sub-seasonal and monthly memory.

    2. **Rolling statistics** (7, 14, 30-day windows):
       rolling_mean — low-frequency trend baseline.
       rolling_std  — local variability / atmospheric instability indicator.
       rolling_min/max — recent extremes, relevant for cold-surge detection.

    3. **Exponential weighted mean** (span=7, span=30):
       Recency-weighted smoothing; tracks rapid temperature transitions.

    4. **Difference features** (diff_1, diff_7):
       Daily tendency and weekly anomaly relative to the same weekday.

    5. **Interaction features**:
       temp_range     = tmax − tmin (diurnal range, cloud-cover proxy).
       humidity_range = umax − umin (boundary-layer moisture gradient).
       pressure_range = p0max − p0min (synoptic pressure gradient).

    6. **Cyclical time encoding** (sin/cos):
       Month (12-month cycle), day_of_year (365.25-day), day_of_week (7-day).
       Encoding as (sin, cos) pairs ensures Dec 31 and Jan 1 are adjacent in
       feature space — critical for learning the annual temperature cycle.

    Parameters
    ----------
    dataframe : pd.DataFrame   Raw meteorological DataFrame with a DatetimeIndex.
    col_target : str           Name of the target column (default: 'tmin').

    Returns
    -------
    pd.DataFrame   Feature-engineered DataFrame with NaN rows dropped.
    """
    df = dataframe.copy()

    # 1. Lag features — causal look-back at multiple timescales
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f'{col_target}_lag{lag}'] = df[col_target].shift(lag)

    # 2. Rolling statistics — .shift(1) before .rolling() enforces causality
    for window in [7, 14, 30]:
        df[f'{col_target}_rolling_mean_{window}'] = df[col_target].shift(1).rolling(window).mean()
        df[f'{col_target}_rolling_std_{window}']  = df[col_target].shift(1).rolling(window).std()
        df[f'{col_target}_rolling_min_{window}']  = df[col_target].shift(1).rolling(window).min()
        df[f'{col_target}_rolling_max_{window}']  = df[col_target].shift(1).rolling(window).max()

    # 3. Exponential weighted mean — recent days weighted more heavily
    df[f'{col_target}_ewm_7']  = df[col_target].shift(1).ewm(span=7).mean()
    df[f'{col_target}_ewm_30'] = df[col_target].shift(1).ewm(span=30).mean()

    # 4. Difference features — day-to-day tendency and weekly anomaly
    df[f'{col_target}_diff_1'] = df[col_target].shift(1).diff(1)
    df[f'{col_target}_diff_7'] = df[col_target].shift(1).diff(7)

    # 5. Interaction / derived meteorological features (all shifted by 1 day)
    df['temp_range']     = df['tmax'].shift(1) - df['tmin'].shift(1)
    df['humidity_range'] = df['umax'].shift(1) - df['umin'].shift(1)
    df['pressure_range'] = df['p0max'].shift(1) - df['p0min'].shift(1)
    # Shift remaining raw covariates to enforce causality
    df['tmax']  = df['tmax'].shift(1)
    df['p0max'] = df['p0max'].shift(1)
    df['p0min'] = df['p0min'].shift(1)
    df['sshn']  = df['sshn'].shift(1)  # sunshine hours — solar forcing proxy
    df['umin']  = df['umin'].shift(1)
    df['umax']  = df['umax'].shift(1)

    # 6. Calendar features → extract integers, then encode cyclically
    df['day_of_year'] = df.index.dayofyear
    df['month']       = df.index.month
    df['day_of_week'] = df.index.dayofweek

    # Cyclical sin/cos encoding — keeps period boundaries continuous
    df['month_sin']       = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']       = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)  # 365.25 handles leap years
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # Drop raw integer calendar columns (encoded versions replace them)
    df.drop(['day_of_year', 'month', 'day_of_week'], axis=1, inplace=True)
    df.dropna(inplace=True)  # remove NaN rows introduced by shift/rolling
    return df


df = create_advanced_features(df)

# Index of the prediction target (tmin) in the full feature matrix.
# Used by the Dataset class to extract ground-truth target vectors.
target_idx = df.columns.get_loc('tmin')

# Cyclical features are already bounded in [-1, 1]; FANLayer handles them separately.
cyclical_features = [
    'month_sin', 'month_cos',
    'day_of_year_sin', 'day_of_year_cos',
    'day_of_week_sin', 'day_of_week_cos'
]
num_cyclical = len(cyclical_features)  # 6

# Convert the DataFrame to a NumPy array once for efficient slicing in Dataset.
cols = list(df.columns)
data = df[cols].values

# ── Chronological Train / Val / Test split (70% / 15% / 15%) ─────────────────
# Strict temporal ordering — no shuffling across splits — prevents future leakage.
train_size = int(0.7 * len(data))
val_size   = int(0.15 * len(data))
train_data = data[:train_size]
val_data   = data[train_size:train_size + val_size]
test_data  = data[train_size + val_size:]

print(f"Train: {train_data.shape},  Val: {val_data.shape},  Test: {test_data.shape}")
print(f"Number of features: {len(cols)}  |  Target index (tmin): {target_idx}")


In [ ]:
# =============================================================================
# FAN-Autoformer — Global Hyperparameters
# =============================================================================

SEQLEN     = 365  # Encoder input length: 1 full year — exposes the complete annual cycle.
LABELLEN   = 182  # Decoder label length: last 6 months of the encoder window
                  # prepended to the decoder for warm-up context.
BATCH_SIZE = 64   # Mini-batch size (uniform across all prediction horizons).

# Prediction horizons to evaluate:
#   H=1  : next-day tmin (operational short-range forecasting)
#   H=3  : 3-day forecast (medium-range synoptic)
#   H=7  : 7-day forecast (sub-seasonal)
#   H=30 : 30-day forecast (monthly climatological outlooks)
PRED_HORIZONS = [1, 3, 7, 30]


## Section 2 — Sliding-Window Dataset

`SlidingWindowDataset` converts the 2-D feature matrix (time × features) into
overlapping **(encoder, decoder, target)** triplets for the Encoder-Decoder architecture.

```
Timeline:  [... t-365 ........... t-182 .............. t-1] [t .. t+pred_len]
Encoder:   [<──────────── seq_len = 365 days ──────────────>]
Decoder:   [               <── label_len=182 ──>][placeholder: pred_len steps]
Target:                                           [ground-truth tmin values  ]
```

The decoder placeholder steps are initialized with the **column-wise mean of
the label segment**, giving the decoder a neutral prior for each feature.
This follows the original Autoformer initialization strategy.


In [ ]:
# =============================================================================
# FAN-Autoformer — Sliding-Window Dataset
# =============================================================================

class SlidingWindowDataset(Dataset):
    """Sliding-window Dataset for FAN-Autoformer's Encoder-Decoder architecture.

    Each sample is a triplet (x_enc, x_dec, y):

    x_enc : FloatTensor (seq_len, n_features)
        Full historical context window fed to the encoder.

    x_dec : FloatTensor (label_len + pred_len, n_features)
        Decoder input:
          x_dec[:label_len] — last label_len steps of x_enc (real historical data).
          x_dec[label_len:] — pred_len placeholder steps initialized to label mean.

    y : FloatTensor (pred_len,)
        Ground-truth tmin values for the forecast horizon.

    Parameters
    ----------
    data : np.ndarray (T, n_features)    Full feature matrix (one temporal split).
    target_col_idx : int                 Column index of tmin in data.
    seq_len : int                        Encoder look-back window (default: 365).
    label_len : int                      Label prefix length for decoder (default: 182).
    pred_len : int                       Forecast horizon in days (default: 1).
    """

    def __init__(self, data, target_col_idx, seq_len=365, label_len=182, pred_len=1):
        self.data            = data
        self.target_col_idx  = target_col_idx
        self.seq_len         = seq_len
        self.label_len       = label_len
        self.pred_len        = pred_len
        sequences, dec_inputs, targets = [], [], []

        for i in range(len(data) - seq_len - pred_len + 1):
            # Encoder input: seq_len historical steps
            seq = data[i : i + seq_len]

            # Decoder label part: last label_len steps of the encoder window
            dec_start_idx = i + seq_len - label_len
            label_part    = data[dec_start_idx : i + seq_len]       # (label_len, F)

            # Decoder placeholder: pred_len steps initialized to label column means
            label_mean = np.mean(label_part, axis=0, keepdims=True)  # (1, F)
            pred_part  = np.repeat(label_mean, pred_len, axis=0)     # (pred_len, F)

            dec_inp = np.concatenate([label_part, pred_part], axis=0)  # (label_len+pred_len, F)

            # Ground-truth target: future tmin values
            target = data[i + seq_len : i + seq_len + pred_len, target_col_idx]

            sequences.append(seq)
            dec_inputs.append(dec_inp)
            targets.append(target)

        # Convert to contiguous NumPy arrays for fast DataLoader iteration
        self.sequences  = np.array(sequences,  dtype=np.float32)
        self.dec_inputs = np.array(dec_inputs, dtype=np.float32)
        self.targets    = np.array(targets,    dtype=np.float32)

    def __len__(self):
        """Return the total number of sliding-window samples."""
        return len(self.sequences)

    def __getitem__(self, idx):
        """Return the (encoder_input, decoder_input, target) triplet for index idx."""
        return (
            torch.FloatTensor(self.sequences[idx]),
            torch.FloatTensor(self.dec_inputs[idx]),
            torch.FloatTensor(self.targets[idx])
        )


## Section 3 — FAN-Autoformer Architecture

```
x_enc (B, 365, F) ──→ FANLayer(norm, cache=True) ──→ Linear(F→d) + pos_enc
                                                                │
                                                    EncoderLayer × 4
                                                    (AutoCorrelation + Decomp + FFN)
                                                                │
                                                            enc_out

x_dec (B, 183, F) ──→ FANLayer(norm, cache=False) ──→ Linear(F→d) + pos_dec
                                                                │
                                                    DecoderLayer × 2
                                                    (Self/Cross AutoCorrelation + Decomp + FFN)
                                                                │
                                                    Linear(d→1)  [last pred_len steps]
                                                                │
                                                    FANLayer(denorm)  [restore °C scale]
                                                                │
                                                    ŷ (B, pred_len)   [tmin forecast]
```


In [ ]:
# =============================================================================
# FAN-Autoformer — FANLayer (Frequency Adaptive Normalization)
# =============================================================================

class FANLayer(nn.Module):
    """Frequency Adaptive Normalization Layer for FAN-Autoformer.

    FAN decomposes each input time series in the **frequency domain**, isolating
    the dominant non-stationary component (low-frequency trends and strong seasonal
    cycles) from the stationary residual (short-term noise).  This is more effective
    than RevIN for meteorological data because it explicitly targets the periodic
    structures — annual cycle, sub-seasonal oscillations — that drive minimum
    temperature variability in semi-arid continental climates such as Ilam Province.

    Normalization pass  (mode='norm')
    ----------------------------------
    1. Real FFT along the time axis → complex frequency coefficients.
    2. Compute mean spectral amplitude across batch and feature dims.
    3. Zero-mask all but the top-K frequency bins (dominant periodic modes).
    4. Inverse FFT → non-stationary component X_nonstat in the time domain.
    5. Return stationary residual  X_res = X_input − X_nonstat.
    6. If context_encoder=True, cache X_nonstat and X_input for later denorm.
    7. Cyclical sin/cos features are passed through unchanged (already in [-1,1]).

    Denormalization pass  (mode='denorm')
    ----------------------------------------
    A two-layer MLP (GELU, Dropout) predicts the future non-stationary shift,
    conditioned on four encoder-side summary statistics:
      [last X_nonstat,  mean X_nonstat,  last X_input,  mean X_input]
    The predicted shift is broadcast over pred_len steps and added back to
    the normalized model output, restoring the original temperature scale.

    Parameters
    ----------
    num_features : int            Total input features (normal + cyclical).
    num_cyclical_features : int   Cyclical features at the tail of the feature axis
                                  (not processed by FFT normalization).
    K : int                       Number of top FFT frequency bins retained (default: 4).
    """

    def __init__(self, num_features, num_cyclical_features=0, K=4):
        super().__init__()
        self.K                     = K
        self.num_features          = num_features
        self.num_cyclical_features = num_cyclical_features
        self.num_normal_features   = num_features - num_cyclical_features

        # MLP for predicting the future non-stationary component during denormalization.
        # Input:  4 × num_normal_features  (last + mean of nonstat and raw input)
        # Output: num_normal_features       (per-feature non-stationary correction)
        self.denorm_mlp = nn.Sequential(
            nn.Linear(self.num_normal_features * 4, 256),
            nn.GELU(),        # smooth non-monotonic activation for continuous signals
            nn.Dropout(0.1),  # regularization against overfitting to encoder statistics
            nn.Linear(256, self.num_normal_features)
        )

        # Cached encoder-side statistics — populated during the norm forward pass
        # (context_encoder=True) and consumed during the denorm pass.
        self.saved_nonstat_nc = None   # non-stationary component of encoder input
        self.saved_input_enc  = None   # raw (non-cyclical) encoder input

    def forward(self, x, mode='norm', context_encoder=False,
                original_x=None, target_idx_in_normal=None):
        """Forward pass of FANLayer.

        Parameters
        ----------
        x : Tensor (batch, seq_len, num_features)
        mode : str   'norm' for normalization, 'denorm' for scale restoration.
        context_encoder : bool   Cache statistics when True (encoder input only).
        original_x : Tensor or None   Retained for API compatibility.
        target_idx_in_normal : int or None
            Index of tmin within the non-cyclical feature slice; used during
            denorm to add back only the target's non-stationary correction.

        Returns
        -------
        Tensor   Normalized residual (norm) or scale-restored prediction (denorm).
        """
        # Separate normal features from the cyclical tail
        if self.num_cyclical_features > 0:
            x_normal   = x[..., :-self.num_cyclical_features]
            x_cyclical = x[..., -self.num_cyclical_features:]
        else:
            x_normal, x_cyclical = x, None

        # ── Normalization ──────────────────────────────────────────────────────
        if mode == 'norm':
            # Step 1: Real FFT along time → complex frequency coefficients
            X_fft    = torch.fft.rfft(x_normal, dim=1)           # (B, freq, F_normal)

            # Step 2: Average spectral amplitude over batch and feature dims
            Amp_mean = torch.abs(X_fft).mean(dim=(0, 2))         # (freq,)

            # Step 3: Keep only the K dominant frequency bins
            _, topk_idx    = torch.topk(Amp_mean, k=min(self.K, len(Amp_mean)))
            X_fft_filtered = torch.zeros_like(X_fft)
            X_fft_filtered[:, topk_idx, :] = X_fft[:, topk_idx, :]

            # Step 4: Inverse FFT → non-stationary component in time domain
            X_nonstat = torch.fft.irfft(X_fft_filtered, n=x_normal.shape[1], dim=1)

            # Step 5: Stationary residual = input minus non-stationary component
            X_res = x_normal - X_nonstat

            # Step 6: Cache encoder statistics for denormalization
            if context_encoder:
                self.saved_nonstat_nc = X_nonstat.detach()
                self.saved_input_enc  = x_normal.detach()

            # Step 7: Reattach unchanged cyclical features
            if x_cyclical is not None:
                return torch.cat([X_res, x_cyclical], dim=-1)
            return X_res

        # ── Denormalization ────────────────────────────────────────────────────
        elif mode == 'denorm':
            # Build the 4-component context vector for the MLP
            nonstat_last = self.saved_nonstat_nc[:, -1, :]      # last step of nonstat
            nonstat_mean = self.saved_nonstat_nc.mean(dim=1)    # mean over encoder time
            input_last   = self.saved_input_enc[:, -1, :]       # last step of raw input
            input_mean   = self.saved_input_enc.mean(dim=1)     # mean of raw input

            mlp_in = torch.cat([nonstat_last, nonstat_mean, input_last, input_mean], dim=-1)
            Y_pred = self.denorm_mlp(mlp_in)                    # (B, F_normal)

            # Broadcast the correction over all pred_len steps
            pred_len = x.shape[1]
            Y_pred   = Y_pred.unsqueeze(1).repeat(1, pred_len, 1)  # (B, pred_len, F_normal)

            # Apply only the target feature's correction when index is known
            if target_idx_in_normal is not None:
                t_ns = Y_pred[:, :, target_idx_in_normal : target_idx_in_normal + 1]
            else:
                t_ns = Y_pred.mean(dim=-1, keepdim=True)  # fallback: average over features

            return x + t_ns  # restore original temperature scale


In [ ]:
# =============================================================================
# FAN-Autoformer — SimpleDecompositionBlock (Dynamic Moving-Average Kernel)
# =============================================================================

class SimpleDecompositionBlock(nn.Module):
    """Moving-average series decomposition for FAN-Autoformer encoder/decoder layers.

    Decomposes the input into:
      - **Trend**:    low-frequency component via symmetric 1-D average pooling.
      - **Seasonal**: high-frequency residual = input − trend.

    Dynamic Kernel Selection
    ------------------------
    The kernel size is set adaptively by the training loop based on pred_len:

        pred_len even →  kernel_size = pred_len + 1   (next odd integer)
        pred_len odd  →  kernel_size = pred_len + 2   (next odd integer above)

    The kernel is always **odd** so that AvgPool1d with symmetric padding preserves
    the sequence length exactly (no off-by-one errors at boundaries).
    Tying the kernel to pred_len aligns the decomposition granularity with the
    forecast horizon, e.g., a 1-day kernel for H=1 vs. a 31-day kernel for H=30.

    Parameters
    ----------
    kernel_size : int   Width of the average-pooling window (must be odd).
    """

    def __init__(self, kernel_size=25):
        super().__init__()
        # AvgPool1d with symmetric padding keeps output length == input length.
        # This is valid ONLY when kernel_size is odd — enforced by the rule above.
        self.moving_avg = nn.AvgPool1d(
            kernel_size=kernel_size,
            stride=1,
            padding=kernel_size // 2
        )

    def forward(self, x):
        """Decompose x into (seasonal, trend).

        Parameters
        ----------
        x : Tensor (batch, seq_len, d_model)

        Returns
        -------
        seasonal : Tensor (batch, seq_len, d_model)   High-frequency residual.
        trend    : Tensor (batch, seq_len, d_model)   Low-frequency trend.
        """
        # AvgPool1d expects (B, channels, L); permute swaps last two axes.
        trend    = self.moving_avg(x.permute(0, 2, 1)).permute(0, 2, 1)
        seasonal = x - trend
        return seasonal, trend


In [ ]:
# =============================================================================
# FAN-Autoformer — AutoCorrelation Mechanism
# =============================================================================

class AutoCorrelation(nn.Module):
    """FFT-based AutoCorrelation attention mechanism (Wu et al., Autoformer 2021).

    Replaces O(L²) dot-product self-attention with O(L log L) time-delay
    aggregation, exploiting the periodic autocorrelation structure of
    meteorological time series.

    Key Insight
    -----------
    Minimum temperature exhibits strong lag-periodicity: today's tmin is highly
    correlated with tmin from 7, 14, 30, and 365 days ago (weekly weather cycles,
    sub-monthly oscillations, and the annual temperature cycle).  AutoCorrelation
    discovers these time delays automatically via FFT cross-correlation.

    Algorithm
    ---------
    For each (Q, K, V) triplet in multi-head format:

    1. Cross-correlation via FFT:
       C(τ) = IFFT( FFT(Q) ⊙ conj(FFT(K)) )
       Equivalent to sliding Q over K and computing dot products at every lag τ,
       but runs in O(L log L) using the convolution theorem.

    2. Select top-k lags:
       Average |C(τ)| over batch and head dims, normalize by √d_model, pick the
       k = ⌊factor·log(L)⌋ lags with the highest absolute correlation.

    3. Weighted aggregation:
       Compute softmax weights over the top-k correlation values; form the output
       as a weighted sum of V circularly shifted (torch.roll) by each selected lag.

    Parameters
    ----------
    d_model : int       Total embedding dimension.
    n_heads : int       Number of attention heads (d_model must be divisible by n_heads).
    dropout : float     Output dropout probability.
    factor  : int       Scaling factor for k: k = max(1, factor·log(L)).
    """

    def __init__(self, d_model, n_heads, dropout=0.1, factor=1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_keys  = d_model // n_heads   # per-head key/query dimension
        self.dropout = nn.Dropout(dropout)
        self.factor  = factor

    def time_delay_agg(self, Q, K, V, topk):
        """Time-delay aggregation via FFT cross-correlation.

        Parameters
        ----------
        Q : Tensor (B, seq_q, n_heads, d_keys)
        K : Tensor (B, seq_k, n_heads, d_keys)
        V : Tensor (B, seq_k, n_heads, d_keys)
        topk : int   Number of lag indices to aggregate.

        Returns
        -------
        Tensor (B, seq_q, n_heads, d_keys)
        """
        B, LQ, H, D = Q.shape
        _, LK, _, _ = K.shape
        L_max = max(LQ, LK)  # pad to equal length before FFT

        # Zero-pad shorter sequences for length-aligned FFT
        if LQ < L_max:
            Q = F.pad(Q, (0, 0, 0, 0, 0, L_max - LQ))
        if LK < L_max:
            K = F.pad(K, (0, 0, 0, 0, 0, L_max - LK))
            V = F.pad(V, (0, 0, 0, 0, 0, L_max - LK))

        # FFT cross-correlation: Q_fft * conj(K_fft) → IFFT → C(τ)
        corr = torch.fft.irfft(
            torch.fft.rfft(Q, dim=1) * torch.conj(torch.fft.rfft(K, dim=1)),
            n=L_max, dim=1
        )   # (B, L_max, n_heads, d_keys)

        # Scalar importance per lag: average over batch and head/key dims
        # Normalize by √d_model (same scaling as standard scaled dot-product attention)
        corr_mean = corr.mean(dim=(0, 2, 3)) / np.sqrt(self.d_model)   # (L_max,)

        # Select top-k lags and compute softmax aggregation weights
        _, tidx  = torch.topk(corr_mean.abs(), k=topk)
        weights  = F.softmax(corr_mean[tidx], dim=0)

        # Weighted circular-shift aggregation:
        # for each selected lag, roll V by that lag and accumulate
        out = torch.zeros_like(Q)
        for i in range(topk):
            out = out + torch.roll(V, shifts=int(tidx[i].item()), dims=1) * weights[i].view(1, 1, 1, 1)

        return out[:, :LQ, :, :]  # remove padding, return original query length

    def forward(self, Q, K, V):
        """Multi-head AutoCorrelation forward pass.

        Parameters
        ----------
        Q : Tensor (B, seq_q, d_model)
        K : Tensor (B, seq_k, d_model)
        V : Tensor (B, seq_k, d_model)

        Returns
        -------
        Tensor (B, seq_q, d_model)
        """
        B, L, _ = Q.shape
        _, S, _ = K.shape
        H = self.n_heads

        # Reshape to multi-head format: (B, seq, n_heads, d_keys)
        Q = Q.view(B, L, H, self.d_keys)
        K = K.view(B, S, H, self.d_keys)
        V = V.view(B, S, H, self.d_keys)

        # k scales logarithmically with sequence length for controlled complexity
        topk = max(1, int(self.factor * np.log(L)))

        out = self.time_delay_agg(Q, K, V, topk=topk)
        return self.dropout(out.reshape(B, L, -1))  # merge heads back to (B, L, d_model)


In [ ]:
# =============================================================================
# FAN-Autoformer — EncoderLayer
# =============================================================================

class EncoderLayer(nn.Module):
    """Single encoder layer of FAN-Autoformer.

    Processing sequence:
      x ──→ AutoCorrelation(x,x,x) ──→ residual + LayerNorm ──→ decomp1
                                                (seasonal only retained)
        ──→ FFN ──→ residual + decomp2 ──→ seasonal output

    Both decomposition steps discard the trend component; the encoder focuses
    on refining the seasonal representation that will be attended to by the decoder.

    Parameters
    ----------
    d_model     : int   Embedding dimension.
    n_heads     : int   AutoCorrelation heads.
    d_ff        : int   FFN hidden dimension.
    dropout     : float Dropout probability.
    kernel_size : int   Odd kernel for both decomposition blocks.
    """

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, kernel_size=25):
        super().__init__()
        self.autocorrelation = AutoCorrelation(d_model, n_heads, dropout)
        self.decomp1 = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.decomp2 = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        """Forward pass through one encoder layer.

        Parameters
        ----------
        x : Tensor (B, seq_len, d_model)

        Returns
        -------
        Tensor (B, seq_len, d_model)  — seasonal component.
        """
        # Block 1: Self-AutoCorrelation + residual + LayerNorm + decompose
        x = self.norm1(x + self.autocorrelation(x, x, x))
        seasonal_after_attn, _ = self.decomp1(x)  # keep seasonal, discard trend

        # Block 2: FFN + residual + decompose (retain seasonal only)
        x = x + self.ff(seasonal_after_attn)
        seasonal_out, _ = self.decomp2(x)
        return seasonal_out


# =============================================================================
# FAN-Autoformer — DecoderLayer
# =============================================================================

class DecoderLayer(nn.Module):
    """Single decoder layer of FAN-Autoformer.

    Three sequential blocks:
      1. Self-AutoCorrelation  — intra-sequence periodic dependencies.
      2. Cross-AutoCorrelation — integrates encoder context (long-range history).
      3. FFN + Decomposition   — non-linear refinement + trend/seasonal separation.

    The accumulated trend from all three decomposition stages is fused via a
    learnable projection and returned alongside the final seasonal component.

    Parameters
    ----------
    d_model     : int   Embedding dimension.
    n_heads     : int   AutoCorrelation heads.
    d_ff        : int   FFN hidden dimension.
    dropout     : float Dropout probability.
    kernel_size : int   Odd kernel for all three decomposition blocks.
    """

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, kernel_size=25):
        super().__init__()
        self.self_corr  = AutoCorrelation(d_model, n_heads, dropout)   # Q=K=V from decoder
        self.cross_corr = AutoCorrelation(d_model, n_heads, dropout)   # Q from dec, K/V from enc

        # Three decomposition blocks — one per processing stage
        self.decomp1 = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.decomp2 = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.decomp3 = SimpleDecompositionBlock(kernel_size=kernel_size)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        # Learnable projection for accumulating multi-level trend components
        self.trend_proj = nn.Linear(d_model, d_model)

    def forward(self, x, enc_out):
        """Forward pass through one decoder layer.

        Parameters
        ----------
        x       : Tensor (B, label_len+pred_len, d_model)  Decoder input.
        enc_out : Tensor (B, seq_len, d_model)              Encoder output.

        Returns
        -------
        s3    : Tensor (B, label_len+pred_len, d_model)  Final seasonal component.
        trend : Tensor (B, label_len+pred_len, d_model)  Accumulated trend.
        """
        # Block 1: Self-AutoCorrelation
        x  = self.norm1(x + self.self_corr(x, x, x))
        s1, _ = self.decomp1(x)

        # Block 2: Cross-AutoCorrelation (s1 as Q, enc_out as K/V)
        s1 = self.norm2(s1 + self.cross_corr(s1, enc_out, enc_out))
        s2, _ = self.decomp2(s1)

        # Block 3: FFN + third decomposition
        s2 = self.norm3(s2 + self.ff(s2))
        s3, _ = self.decomp3(s2)

        # Accumulate trend from all three seasonal stages via learnable projection
        trend = s1 + self.trend_proj(s2) + self.trend_proj(s3)
        return s3, trend


In [ ]:
# =============================================================================
# FAN-Autoformer — Main Model Class
# =============================================================================

class Autoformer(nn.Module):
    """FAN-Autoformer: Autoformer with Frequency Adaptive Normalization and
    adaptive moving-average decomposition for meteorological forecasting.

    This model is the primary contribution of the paper.  It combines:
    - **FANLayer**: frequency-domain non-stationarity removal at input/output.
    - **SimpleDecompositionBlock** with adaptive odd kernel: trend/seasonal
      separation aligned with the prediction horizon.
    - **AutoCorrelation**: O(L log L) FFT-based time-delay attention.

    Forward Pass
    ------------
    1. FANLayer normalizes the encoder input, caching non-stationary statistics.
    2. Linear embedding + learnable positional encoding.
    3. Four EncoderLayers progressively refine the seasonal representation.
    4. FANLayer normalizes the decoder input (context not updated).
    5. Two DecoderLayers: self + cross AutoCorrelation with decomposition.
    6. Seasonal and trend outputs are summed with the decoder state.
    7. Linear projection maps the last pred_len steps to scalar forecasts.
    8. FANLayer denormalization restores the original temperature scale.

    Parameters
    ----------
    n_features          : int    Total input features (normal + cyclical).
    d_model             : int    Embedding dimension (default: 64).
    n_heads             : int    AutoCorrelation heads (default: 4).
    n_encoder_layers    : int    Encoder depth (default: 4).
    n_decoder_layers    : int    Decoder depth (default: 2).
    d_ff                : int    FFN hidden dimension (default: 256).
    seq_len             : int    Encoder input length (default: 365).
    pred_len            : int    Forecast horizon (1, 3, 7, or 30).
    dropout             : float  Dropout probability (default: 0.15).
    num_cyclical_features: int   Cyclical features at tail of feature axis.
    fan_K               : int    Top-K FFT frequency bins in FANLayer.
    target_idx          : int    Column index of tmin in the feature matrix.
    moving_avg_kernel   : int    Decomposition kernel (odd, adaptive per horizon).
    """

    def __init__(self, n_features, d_model, n_heads, n_encoder_layers, n_decoder_layers,
                 d_ff, seq_len, pred_len, dropout,
                 num_cyclical_features=0, fan_K=4, target_idx=None,
                 moving_avg_kernel=25):
        super().__init__()
        self.pred_len = pred_len

        # Map target_idx to its position within the non-cyclical feature slice
        # (FANLayer denorm operates only on non-cyclical features)
        self.target_idx_in_normal = (
            target_idx
            if target_idx is not None and target_idx < n_features - num_cyclical_features
            else None
        )

        # Shared FANLayer for encoder norm, decoder norm, and output denorm
        self.fan_layer = FANLayer(n_features, num_cyclical_features, K=fan_K)

        # Separate linear embeddings for encoder and decoder inputs
        self.enc_embedding = nn.Linear(n_features, d_model)
        self.dec_embedding = nn.Linear(n_features, d_model)

        # Learnable positional encodings — small-std Gaussian init, learned end-to-end
        self.pos_enc = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.pos_dec = nn.Parameter(torch.randn(1, seq_len + pred_len, d_model) * 0.02)

        # Encoder stack: 4 layers, each with adaptive kernel decomposition
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout, kernel_size=moving_avg_kernel)
            for _ in range(n_encoder_layers)
        ])

        # Decoder stack: 2 layers, each with adaptive kernel decomposition
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout, kernel_size=moving_avg_kernel)
            for _ in range(n_decoder_layers)
        ])

        # Final projection: d_model → 1 scalar (normalized tmin prediction)
        self.projection = nn.Linear(d_model, 1)

    def forward(self, x_enc, x_dec):
        """FAN-Autoformer forward pass.

        Parameters
        ----------
        x_enc : Tensor (B, seq_len, n_features)          One year of meteorological history.
        x_dec : Tensor (B, label_len+pred_len, n_features) Label + placeholder decoder input.

        Returns
        -------
        Tensor (B, pred_len)   Predicted daily minimum temperature (°C).
        """
        # ── Encoder ────────────────────────────────────────────────────────────
        # FANLayer normalizes and caches non-stationary statistics for denorm
        enc_out = self.enc_embedding(
            self.fan_layer(x_enc, mode='norm', context_encoder=True)
        ) + self.pos_enc[:, :x_enc.size(1), :]

        for layer in self.encoder_layers:
            enc_out = layer(enc_out)   # iteratively refine seasonal representation

        # ── Decoder ────────────────────────────────────────────────────────────
        # Normalize decoder input without overwriting the encoder cache
        dec_out = self.dec_embedding(
            self.fan_layer(x_dec, mode='norm', context_encoder=False)
        ) + self.pos_dec[:, :x_dec.size(1), :]

        seasonal_part = trend_part = 0
        for layer in self.decoder_layers:
            seasonal_part, trend_part = layer(dec_out, enc_out)

        # Fuse seasonal and trend outputs into the decoder state
        dec_out = dec_out + seasonal_part + trend_part

        # ── Output projection + denormalization ───────────────────────────────
        output = self.projection(dec_out[:, -self.pred_len:, :])   # (B, pred_len, 1)

        # FANLayer restores the original temperature scale using cached encoder stats
        return self.fan_layer(
            output, mode='denorm', context_encoder=True,
            original_x=x_enc,
            target_idx_in_normal=self.target_idx_in_normal
        ).squeeze(-1)   # (B, pred_len)


In [ ]:
# =============================================================================
# FAN-Autoformer — LayerWiseInterpreter (Gradient-Based XAI)
# =============================================================================

class LayerWiseInterpreter:
    """Layer-wise explainability module for FAN-Autoformer.

    Registers PyTorch forward and backward hooks on every AutoCorrelation and
    FFN sub-module in the encoder and decoder stacks.  After each backward pass,
    the stored activations and gradients are combined into feature importance
    maps using a gradient × activation attribution metric inspired by Grad-CAM.

    Importance Metric
    -----------------
    For hooked layer L and embedding dimension d:

        importance[d] = mean over (batch, time) of |activation[d]| × |gradient[d]|

    This first-order Taylor approximation estimates how much the loss would change
    if feature dimension d were zeroed at layer L.  The result is min-max normalized
    to [0, 1] and mapped to the input feature names via tile/truncate alignment.

    Hooked Modules
    --------------
    encoder_attn_{i}       encoder_layers[i].autocorrelation
    encoder_ff_{i}         encoder_layers[i].ff
    decoder_self_attn_{i}  decoder_layers[i].self_corr
    decoder_cross_attn_{i} decoder_layers[i].cross_corr
    decoder_ff_{i}         decoder_layers[i].ff

    Parameters
    ----------
    model         : Autoformer     Fully constructed FAN-Autoformer instance.
    feature_names : list of str    Human-readable names for the n_features inputs.
    device        : str            'cpu' or 'cuda'.
    """

    def __init__(self, model, feature_names, device='cpu'):
        self.model             = model
        self.feature_names     = feature_names
        self.device            = device
        self.layer_activations = {}   # forward-hook cache: layer_name → activation tensor
        self.layer_gradients   = {}   # backward-hook cache: layer_name → gradient tensor
        self.input_data        = None # last encoder input batch (for saliency analysis)
        self.register_hooks()

    def register_hooks(self):
        """Register forward and full_backward hooks on all attention and FFN sub-modules."""
        # Encoder hooks
        for idx, layer in enumerate(self.model.encoder_layers):
            layer.autocorrelation.register_forward_hook(
                self.create_activation_hook(f'encoder_attn_{idx}'))
            layer.autocorrelation.register_full_backward_hook(
                self.create_gradient_hook(f'encoder_attn_{idx}'))
            layer.ff.register_forward_hook(
                self.create_activation_hook(f'encoder_ff_{idx}'))
            layer.ff.register_full_backward_hook(
                self.create_gradient_hook(f'encoder_ff_{idx}'))

        # Decoder hooks
        for idx, layer in enumerate(self.model.decoder_layers):
            layer.self_corr.register_forward_hook(
                self.create_activation_hook(f'decoder_self_attn_{idx}'))
            layer.self_corr.register_full_backward_hook(
                self.create_gradient_hook(f'decoder_self_attn_{idx}'))
            layer.cross_corr.register_forward_hook(
                self.create_activation_hook(f'decoder_cross_attn_{idx}'))
            layer.cross_corr.register_full_backward_hook(
                self.create_gradient_hook(f'decoder_cross_attn_{idx}'))
            layer.ff.register_forward_hook(
                self.create_activation_hook(f'decoder_ff_{idx}'))
            layer.ff.register_full_backward_hook(
                self.create_gradient_hook(f'decoder_ff_{idx}'))

    def create_activation_hook(self, name):
        """Return a forward hook that caches the layer output tensor."""
        def hook(module, input, output):
            self.layer_activations[name] = output.detach()  # detach prevents graph retention
        return hook

    def create_gradient_hook(self, name):
        """Return a backward hook that caches the gradient of the layer output."""
        def hook(module, grad_input, grad_output):
            if grad_output[0] is not None:
                self.layer_gradients[name] = grad_output[0].detach()
        return hook

    def store_input(self, x_enc):
        """Cache the encoder input batch for gradient saliency analysis."""
        self.input_data = x_enc.detach()

    def compute_feature_importance(self):
        """Compute per-layer feature importance maps from cached hooks.

        Returns
        -------
        dict  Keys: layer names (str).  Values: dicts with:
              'importance'  : np.ndarray (n_features,), values in [0, 1].
              'top_features': list of (feature_name, score) for top-5 features.
        """
        if not self.layer_activations or not self.layer_gradients:
            return {}

        layer_importance = {}
        for layer_name in sorted(self.layer_activations.keys()):
            if layer_name not in self.layer_gradients:
                continue
            activation = self.layer_activations[layer_name]
            gradient   = self.layer_gradients[layer_name]

            # Activation × gradient, averaged over batch and time → importance per dim
            importance_map = (gradient.abs() * activation.abs()).mean(dim=(0, 1))

            # Min-max normalize to [0, 1] for cross-layer comparability
            if importance_map.max() > 0:
                importance_map = importance_map / importance_map.max()

            importance_np = importance_map.cpu().numpy()

            # Align length to n_features: tile if shorter, truncate if longer
            if len(importance_np) < len(self.feature_names):
                repeat_factor = int(np.ceil(len(self.feature_names) / len(importance_np)))
                importance_np = np.tile(importance_np, repeat_factor)[:len(self.feature_names)]
            elif len(importance_np) > len(self.feature_names):
                importance_np = importance_np[:len(self.feature_names)]

            if len(importance_np) == len(self.feature_names):
                top_k       = min(5, len(self.feature_names))
                top_indices = np.argsort(importance_np)[-top_k:][::-1]
                layer_importance[layer_name] = {
                    'importance':   importance_np,
                    'top_features': [
                        (self.feature_names[i], float(importance_np[i]))
                        for i in top_indices
                    ]
                }
        return layer_importance

    def compute_input_gradient_importance(self):
        """Aggregate gradient magnitudes across all hooked layers as a saliency proxy.

        Returns
        -------
        np.ndarray (n_features,) or None
        """
        try:
            if self.input_data is None:
                return None
            all_gradients = [
                grad.abs().mean(dim=1).squeeze(0)
                for grad in self.layer_gradients.values()
            ]
            if all_gradients:
                avg_gradient = torch.stack(all_gradients).mean(dim=0).cpu().numpy()
                if len(avg_gradient) < len(self.feature_names):
                    return np.tile(avg_gradient,
                                   max(1, len(self.feature_names) // len(avg_gradient))
                                   )[:len(self.feature_names)]
                elif len(avg_gradient) == len(self.feature_names):
                    return avg_gradient
        except Exception as e:
            print(f"⚠️  Warning in compute_input_gradient_importance: {e}")
        return None

    def clear_cache(self):
        """Clear all cached activations, gradients, and the stored input batch.
        Must be called before each dedicated interpretability forward/backward pass."""
        self.layer_activations.clear()
        self.layer_gradients.clear()
        self.input_data = None


In [ ]:
# =============================================================================
# FAN-Autoformer — AdvancedRealtimeVisualizer (Plotly XAI Dashboards)
# =============================================================================

class AdvancedRealtimeVisualizer:
    """Interactive visualization engine for FAN-Autoformer interpretability output.

    Produces four Plotly HTML dashboards saved to disk during and after training:

    Dashboard 1 — Training Progress
        Line chart of validation MAE vs. epoch with a vertical cursor at the
        current epoch.  Monitors convergence and early-stopping behaviour.

    Dashboard 2 — Attention Heatmap
        Layer × top-10-feature importance matrix for encoder and decoder,
        displayed as a color-coded heatmap.  Reveals which input features are
        most influential at each network depth.

    Dashboard 3 — Layer-Wise Importance Bar Chart
        Horizontal bar chart (one subplot per layer, up to 6) showing the top-5
        most important features at the most recent snapshot.  For more than 6
        layers, all are grouped in a single faceted bar chart.

    Dashboard 4 — Temporal Feature Importance Trend
        Multi-line chart of the top-10 most important features over training
        epochs.  Tracks whether feature relevance is stable, increasing, or
        collapsing as the model trains.

    Snapshot History
    ----------------
    Each snapshot: { 'epoch', 'importance', 'input_importance', 'val_mae' }.
    Appended via add_snapshot() once per epoch.

    Parameters
    ----------
    feature_names : list of str    Input feature names (same as LayerWiseInterpreter).
    save_dir      : str or None    Directory for HTML output files.
    """

    def __init__(self, feature_names, save_dir=None):
        self.feature_names = feature_names
        self.history       = []
        self.save_dir      = save_dir

    def add_snapshot(self, epoch, layer_importance, input_importance=None, val_mae=None):
        """Append an interpretability snapshot to the history buffer."""
        if not layer_importance:
            return
        self.history.append({
            'epoch':            epoch,
            'importance':       layer_importance,
            'input_importance': input_importance,
            'val_mae':          val_mae,
        })

    # ── Dashboard 4b: All-Layer Feature Importance Evolution ──────────────────
    def visualize_all_layers_evolution(self, top_k=10):
        """Multi-panel line chart of per-layer feature importance evolution over snapshots."""
        if not self.history:
            return None
        all_importances = {}
        for snapshot in self.history:
            for layer_name, imp_data in snapshot['importance'].items():
                all_importances.setdefault(layer_name, {})
                for feat_name, imp_score in imp_data['top_features']:
                    all_importances[layer_name].setdefault(feat_name, []).append(imp_score)
        layer_names = sorted(all_importances.keys())
        n_layers    = len(layer_names)
        if n_layers == 0:
            return None
        vertical_spacing = min(0.05, (1.0 / (n_layers - 1)) * 0.8) if n_layers > 1 else 0.05
        fig = make_subplots(rows=n_layers, cols=1,
                            subplot_titles=[f"📊 {n}" for n in layer_names],
                            vertical_spacing=vertical_spacing)
        colors = ['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A','#98D8C8',
                  '#F7DC6F','#BB8FCE','#85C1E2','#F8B88B','#99CC99']
        for li, layer_name in enumerate(layer_names):
            layer_data     = all_importances[layer_name]
            avg_importance = {f: np.mean(s) for f, s in layer_data.items()}
            top_feats      = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)[:top_k]
            for fi, (feat_name, _) in enumerate(top_feats):
                if feat_name in layer_data:
                    fig.add_trace(go.Scatter(
                        x=list(range(len(layer_data[feat_name]))),
                        y=layer_data[feat_name], mode='lines+markers',
                        name=feat_name,
                        line=dict(color=colors[fi % len(colors)], width=2),
                        marker=dict(size=4), showlegend=(li == 0)
                    ), row=li + 1, col=1)
            fig.update_xaxes(title_text="Snapshot", row=li + 1, col=1)
            fig.update_yaxes(title_text="Importance", row=li + 1, col=1)
        total_height = max(200, min(300, 3000 // n_layers)) * n_layers
        fig.update_layout(
            title="🎯 FAN-Autoformer — Feature Importance Evolution (All Layers)",
            font=dict(family="B Lotus,Arial", size=14),
            height=total_height, template='plotly_white',
            hovermode='x unified', showlegend=True)
        return fig

    # ── Dashboard 3: Layer-Wise Importance Bar Chart ───────────────────────────
    def visualize_layer_importance_comparison(self):
        """Horizontal bar chart of top-5 features per layer (most recent snapshot)."""
        if not self.history:
            return None
        layer_importance = self.history[-1]['importance']
        layer_names      = sorted(layer_importance.keys())
        if not layer_names:
            return None
        max_layers_per_fig = 6
        if len(layer_names) <= max_layers_per_fig:
            fig = make_subplots(rows=1, cols=len(layer_names),
                                subplot_titles=[f"📈 {n}" for n in layer_names],
                                specs=[[{'type': 'bar'}]*len(layer_names)],
                                horizontal_spacing=0.05)
            for ci, layer_name in enumerate(layer_names):
                imp_data     = layer_importance[layer_name]
                top_features = [f[0] for f in imp_data['top_features']]
                top_values   = [f[1] for f in imp_data['top_features']]
                fig.add_trace(go.Bar(
                    y=top_features, x=top_values, orientation='h',
                    name=layer_name, showlegend=False,
                    marker=dict(color=top_values, colorscale='Viridis', showscale=(ci == 0)),
                    text=[f'{v:.3f}' for v in top_values], textposition='auto'
                ), row=1, col=ci + 1)
                fig.update_xaxes(title_text="Score", row=1, col=ci + 1)
            fig.update_layout(
                title="🏆 FAN-Autoformer — Feature Importance Comparison Across Layers",
                font=dict(family="B Lotus,Arial", size=14),
                height=500, template='plotly_white', hovermode='y')
        else:
            fig = go.Figure()
            for layer_name in layer_names:
                imp_data     = layer_importance[layer_name]
                top_features = [f[0] for f in imp_data['top_features']]
                top_values   = [f[1] for f in imp_data['top_features']]
                fig.add_trace(go.Bar(
                    y=top_features, x=top_values, orientation='h',
                    name=layer_name,
                    text=[f'{v:.3f}' for v in top_values], textposition='auto'))
            fig.update_layout(
                title="🏆 FAN-Autoformer — Feature Importance (All Layers)",
                font=dict(family="B Lotus,Arial", size=14),
                xaxis_title="Importance Score", yaxis_title="Features",
                height=600, template='plotly_white', barmode='group', showlegend=True)
        return fig

    # ── Dashboard 2: Encoder / Decoder Attention Heatmap ─────────────────────
    def visualize_attention_heatmap(self):
        """Layer × top-10-feature importance heatmap for encoder and decoder."""
        if not self.history:
            return None
        layer_importance = self.history[-1]['importance']
        encoder_layers   = {k: v for k, v in layer_importance.items() if 'encoder' in k}
        decoder_layers   = {k: v for k, v in layer_importance.items() if 'decoder' in k}
        if not encoder_layers and not decoder_layers:
            return None
        n_cols = (1 if encoder_layers else 0) + (1 if decoder_layers else 0)
        fig = make_subplots(
            rows=1, cols=n_cols,
            subplot_titles=[s for s in ["🔴 Encoder Layers", "🔵 Decoder Layers"]
                            if (encoder_layers if "Encoder" in s else decoder_layers)],
            specs=[[{'type': 'heatmap'}]*n_cols])
        top_feats = self.feature_names[:10]
        col_idx   = 1

        def build_matrix(layer_dict):
            layer_list = sorted(layer_dict.keys())
            matrix = []
            for layer in layer_list:
                row = []
                for feat in top_feats:
                    val = next((score for name, score in layer_dict[layer]['top_features'] if name == feat), 0.0)
                    row.append(val)
                matrix.append(row)
            return layer_list, matrix

        if encoder_layers:
            enc_names, enc_matrix = build_matrix(encoder_layers)
            fig.add_trace(go.Heatmap(z=enc_matrix, y=enc_names, x=top_feats,
                                     colorscale='RdYlBu_r', showscale=True, name='Encoder'),
                          row=1, col=col_idx)
            col_idx += 1
        if decoder_layers:
            dec_names, dec_matrix = build_matrix(decoder_layers)
            fig.add_trace(go.Heatmap(z=dec_matrix, y=dec_names, x=top_feats,
                                     colorscale='RdYlBu_r', showscale=False, name='Decoder'),
                          row=1, col=col_idx)
        fig.update_layout(
            title="🔥 FAN-Autoformer — Attention Heatmap (Layer × Feature Importance)",
            font=dict(family="B Lotus,Arial", size=14),
            height=max(400, len(encoder_layers)*50 + len(decoder_layers)*50 + 100),
            template='plotly_white')
        return fig

    # ── Dashboard 4: Temporal Feature Importance Trend ────────────────────────
    def visualize_temporal_importance_trend(self):
        """Multi-line chart of top-10 feature importance over training epochs."""
        if not self.history or len(self.history) < 2:
            return None
        feature_trends = {feat: [] for feat in self.feature_names}
        epochs = []
        for snapshot in self.history:
            epochs.append(snapshot['epoch'])
            for feat in self.feature_names:
                scores = [score for imp_data in snapshot['importance'].values()
                          for name, score in imp_data['top_features'] if name == feat]
                feature_trends[feat].append(np.mean(scores) if scores else 0.0)
        top_features = sorted(
            [(f, np.mean(s)) for f, s in feature_trends.items()],
            key=lambda x: x[1], reverse=True
        )[:10]
        colors = ['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A','#98D8C8',
                  '#F7DC6F','#BB8FCE','#85C1E2','#F8B88B','#99CC99']
        fig = go.Figure()
        for idx, (feat, _) in enumerate(top_features):
            fig.add_trace(go.Scatter(
                x=epochs, y=feature_trends[feat], mode='lines+markers',
                name=feat, line=dict(color=colors[idx], width=3), marker=dict(size=6)))
        fig.update_layout(
            title="📈 FAN-Autoformer — Temporal Feature Importance Trend",
            font=dict(family="B Lotus,Arial", size=14),
            xaxis_title="Epoch", yaxis_title="Average Importance Score",
            height=600, template='plotly_white', hovermode='x unified', showlegend=True)
        return fig


In [ ]:
# =============================================================================
# FAN-Autoformer — Evaluation Utilities
# =============================================================================

def calculate_smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error (sMAPE) in percent.

    Scale-invariant and bounded — suitable for comparing forecast quality
    across horizons where absolute error magnitude naturally grows with pred_len.

    Formula: sMAPE = 100 × mean( |y_true − y_pred| / ((|y_true| + |y_pred|) / 2) )

    A small epsilon (1e-8) prevents division by zero.
    """
    return np.mean(
        np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8)
    ) * 100


def evaluate_model(model, dataloader, device, criterion=None):
    """Evaluate FAN-Autoformer on an entire DataLoader split.

    Parameters
    ----------
    model      : Autoformer          Trained model instance.
    dataloader : DataLoader          Val or test DataLoader (shuffle=False).
    device     : torch.device        Compute device.
    criterion  : nn.Module or None   Optional loss function.

    Returns
    -------
    dict with keys:
        'mae'           : Mean Absolute Error (°C)
        'rmse'          : Root Mean Squared Error (°C)
        'r2_score'      : Coefficient of determination R²
        'smape_percent' : sMAPE (%)
        'loss'          : Mean criterion loss (None if not provided)
        'preds'         : np.ndarray of all predictions
        'targets'       : np.ndarray of all ground-truth values
    """
    model.eval()
    all_preds, all_targets, total_loss = [], [], 0.0

    with torch.no_grad():
        for x_enc, x_dec, y in dataloader:
            x_enc, x_dec, y = x_enc.to(device), x_dec.to(device), y.to(device)
            out = model(x_enc, x_dec)
            if criterion is not None:
                total_loss += criterion(out, y).item()
            all_preds.append(out.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    preds   = np.concatenate(all_preds).flatten()
    targets = np.concatenate(all_targets).flatten()

    return {
        'mae':           mean_absolute_error(targets, preds),
        'rmse':          np.sqrt(mean_squared_error(targets, preds)),
        'r2_score':      r2_score(targets, preds),
        'smape_percent': calculate_smape(targets, preds),
        'loss':          total_loss / len(dataloader) if criterion is not None else None,
        'preds':         preds,
        'targets':       targets
    }


In [ ]:
# =============================================================================
# FAN-Autoformer — Training Loop with Real-Time Interpretability
# =============================================================================

def train_with_interpretability(model, train_loader, val_loader, feature_names,
                                epochs=50, lr=5e-4, device='cpu',
                                seed=None, pred_len=1, verbose=False):
    """Train FAN-Autoformer with per-epoch XAI snapshots and Plotly dashboards.

    Training Configuration
    ----------------------
    Optimizer : AdamW (weight_decay=1e-5)
        Decoupled weight decay regularizes the model without penalizing biases.
    Scheduler : OneCycleLR (pct_start=0.1, cosine annealing)
        10% warm-up phase followed by smooth cosine decay — consistently
        accelerates convergence compared to fixed or step-decay schedules.
    Loss      : MSELoss
        Penalizes large temperature errors quadratically, suitable for a
        regression task where extreme cold events carry high operational cost.
    Early stopping : patience=20 epochs on validation MAE.

    Interpretability Integration
    ----------------------------
    At the end of each epoch, the model runs a dedicated forward/backward pass
    on the last batch (after clearing hook caches) to capture clean activation
    and gradient signals.  A snapshot is added to AdvancedRealtimeVisualizer,
    and four Plotly HTML files are written every 5 epochs and at the final epoch.

    Output directory: interpretability_outputs/seed{seed}_pred{pred_len}/

    Parameters
    ----------
    model         : Autoformer     Freshly initialized FAN-Autoformer instance.
    train_loader  : DataLoader     Training split (shuffle=True, drop_last=True).
    val_loader    : DataLoader     Validation split (shuffle=False).
    feature_names : list of str    Full feature list including target column.
    epochs        : int            Maximum training epochs (default: 50).
    lr            : float          Peak learning rate for OneCycleLR (default: 5e-4).
    device        : torch.device   Compute device.
    seed          : int or None    Random seed (for checkpoint naming).
    pred_len      : int            Prediction horizon (for checkpoint naming).
    verbose       : bool           Print epoch metrics every 5 epochs.

    Returns
    -------
    model : Autoformer   Model at last epoch (load the saved .pth for best weights).
    """
    import os
    os.makedirs('interpretability_outputs', exist_ok=True)

    # ── Optimizer and LR scheduler ────────────────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,        # 10% warm-up
        anneal_strategy='cos' # smooth cosine decay after warm-up
    )
    criterion = nn.MSELoss()

    # ── Interpretability tools ─────────────────────────────────────────────────
    interpreter = LayerWiseInterpreter(model, feature_names=feature_names, device=device)
    interpreter.register_hooks()   # register hooks after optimizer is assigned
    save_dir   = f'interpretability_outputs/seed{seed}_pred{pred_len}'
    visualizer = AdvancedRealtimeVisualizer(feature_names=feature_names, save_dir=save_dir)

    # ── Early stopping state ──────────────────────────────────────────────────
    best_val_mae = float('inf')
    patience_ctr = 0
    patience     = 20

    # ── Main training loop ────────────────────────────────────────────────────
    for epoch in range(epochs):
        model.train()
        total_loss       = 0.0
        epoch_importance = {}   # updated at the end-of-epoch interpretability pass

        for batch_idx, (x_enc, x_dec, y) in enumerate(train_loader):
            x_enc, x_dec, y = x_enc.to(device), x_dec.to(device), y.to(device)
            interpreter.store_input(x_enc)

            optimizer.zero_grad()
            out  = model(x_enc, x_dec)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

            # End-of-epoch interpretability pass: clear caches, re-run forward/backward,
            # extract importance maps from clean hook captures.
            if batch_idx == len(train_loader) - 1:
                interpreter.clear_cache()
                out_interp       = model(x_enc, x_dec)
                loss_interp      = criterion(out_interp, y)
                loss_interp.backward()
                epoch_importance = interpreter.compute_feature_importance()
                input_importance = interpreter.compute_input_gradient_importance()

        # ── Validation ─────────────────────────────────────────────────────────
        val_res   = evaluate_model(model, val_loader, device, criterion)
        val_mae   = val_res['mae']
        val_smape = val_res['smape_percent']

        # ── Snapshot and periodic Plotly dashboard export ─────────────────────
        if epoch_importance:
            visualizer.add_snapshot(epoch, epoch_importance, input_importance, val_mae)

            if epoch % 5 == 0 or epoch == epochs - 1:
                if visualizer.save_dir:
                    os.makedirs(visualizer.save_dir, exist_ok=True)

                    # Dashboard 2: Attention heatmap
                    fig = visualizer.visualize_attention_heatmap()
                    if fig:
                        fig.write_html(f'{visualizer.save_dir}/attention_heatmap_epoch{epoch+1}.html')

                    # Dashboard 3: Layer-wise importance bars
                    fig = visualizer.visualize_layer_importance_comparison()
                    if fig:
                        fig.write_html(f'{visualizer.save_dir}/layer_importance_epoch{epoch+1}.html')

                    if len(visualizer.history) >= 2:
                        # Dashboard 4: Temporal trend (requires ≥ 2 snapshots)
                        fig = visualizer.visualize_temporal_importance_trend()
                        if fig:
                            fig.write_html(f'{visualizer.save_dir}/temporal_trend_epoch{epoch+1}.html')

                        # Dashboard 4b: All-layers evolution
                        fig = visualizer.visualize_all_layers_evolution()
                        if fig:
                            fig.write_html(f'{visualizer.save_dir}/all_layers_evolution_epoch{epoch+1}.html')

                    # Dashboard 1: Training progress
                    val_maes = [s['val_mae'] for s in visualizer.history if s.get('val_mae') is not None]
                    if val_maes:
                        fig_progress = go.Figure()
                        fig_progress.add_trace(go.Scatter(
                            x=list(range(len(val_maes))), y=val_maes,
                            mode='lines+markers', name='Val MAE',
                            line=dict(color='blue', width=2), marker=dict(size=4)))
                        fig_progress.add_vline(
                            x=len(val_maes) - 1, line_dash='dash',
                            line_color='red', opacity=0.5,
                            annotation_text=f'Epoch {epoch+1}')
                        fig_progress.update_layout(
                            title=f'FAN-Autoformer Training Progress — Seed={seed}  PredLen={pred_len}  Epoch {epoch+1}',
                            xaxis_title='Epoch', yaxis_title='Validation MAE (°C)',
                            template='plotly_white',
                            font=dict(family='B Lotus,Arial', size=14), height=400)
                        fig_progress.write_html(f'{visualizer.save_dir}/training_progress_epoch{epoch+1}.html')

        # ── Verbose logging ────────────────────────────────────────────────────
        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  Epoch {epoch+1:3d} | "
                  f"Train MSE: {total_loss/len(train_loader):.4f} | "
                  f"Val MAE: {val_mae:.4f}°C | "
                  f"Val sMAPE: {val_smape:.4f}%")

        # ── Early stopping and best-model checkpoint ───────────────────────────
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_ctr = 0
            torch.save(
                {'model_state_dict': model.state_dict(), 'val_mae': best_val_mae},
                f'best_autoformer_seed{seed}_pred{pred_len}.pth'
            )
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                if verbose:
                    print(f"  ⏹  Early stopping at epoch {epoch+1} "
                          f"(no improvement for {patience} consecutive epochs)")
                break

    # ── Final dashboard export after training completes ───────────────────────
    if visualizer.save_dir:
        for name, method in [
            ('temporal_trend_final',      visualizer.visualize_temporal_importance_trend),
            ('all_layers_evolution_final', visualizer.visualize_all_layers_evolution),
            ('attention_heatmap_final',    visualizer.visualize_attention_heatmap),
            ('layer_importance_final',     visualizer.visualize_layer_importance_comparison),
        ]:
            fig = method()
            if fig:
                fig.write_html(f'{visualizer.save_dir}/{name}.html')

    return model


In [ ]:
# =============================================================================
# FAN-Autoformer — Main Experiment Loop (Multi-Horizon × Multi-Seed)
# =============================================================================

if __name__ == "__main__":

    seeds  = [42, 123, 456, 789, 1011]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Compute device: {device}")

    all_results = []  # flat list of per-run metric dicts; written to CSV at the end

    for pred_len in PRED_HORIZONS:
        print(f"\n{'#'*62}")
        print(f"#  FAN-Autoformer — Prediction Horizon: {pred_len} day(s)")
        print(f"{'#'*62}")

        # ── Dynamic kernel selection ──────────────────────────────────────────
        # The moving-average kernel must be odd (AvgPool1d symmetric-padding rule).
        # The rule selects the smallest odd integer strictly greater than pred_len,
        # aligning decomposition smoothness with the forecast horizon scale.
        if pred_len % 2 == 0:
            kernel_size = pred_len + 1   # even pred_len → next odd integer
        else:
            kernel_size = pred_len + 2   # odd pred_len  → skip to stay odd
        print(f"🔧 Dynamic kernel_size = {kernel_size}  (pred_len={pred_len})")

        # ── Horizon-specific DataLoaders (built once, reused across seeds) ────
        train_ds = SlidingWindowDataset(train_data, target_idx, SEQLEN, LABELLEN, pred_len)
        val_ds   = SlidingWindowDataset(val_data,   target_idx, SEQLEN, LABELLEN, pred_len)
        test_ds  = SlidingWindowDataset(test_data,  target_idx, SEQLEN, LABELLEN, pred_len)

        horizon_results = []  # metric dicts for all seeds at this horizon

        for seed in seeds:
            print(f"\n{'='*62}")
            print(f"FAN-Autoformer | seed={seed} | pred_len={pred_len} | kernel={kernel_size}")

            # ── Deterministic seed initialization ─────────────────────────────
            # All random number generators are seeded identically to guarantee
            # fully reproducible weight initialization and mini-batch ordering.
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark     = False
            os.environ['PYTHONHASHSEED'] = str(seed)
            torch.cuda.empty_cache()
            gc.collect()

            # ── Instantiate FAN-Autoformer ────────────────────────────────────
            model = Autoformer(
                n_features            = len(df.columns),  # total feature count (incl. target)
                d_model               = 64,               # embedding dimension
                n_heads               = 4,                # AutoCorrelation heads (64/4=16 per head)
                n_encoder_layers      = 4,                # encoder stack depth
                n_decoder_layers      = 2,                # decoder stack depth
                d_ff                  = 256,              # FFN hidden dimension
                seq_len               = SEQLEN,           # 365-day encoder look-back
                pred_len              = pred_len,
                dropout               = 0.15,
                num_cyclical_features = num_cyclical,
                fan_K                 = 4,                # top-4 FFT components in FANLayer
                target_idx            = target_idx,       # tmin column index
                moving_avg_kernel     = kernel_size       # horizon-adaptive odd kernel
            ).to(device)
            print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

            # ── DataLoaders ────────────────────────────────────────────────────
            train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True)
            val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False)
            test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False)

            # ── Train ──────────────────────────────────────────────────────────
            t0    = time.perf_counter()
            model = train_with_interpretability(
                model, train_loader, val_loader,
                feature_names = list(df.columns),
                epochs        = 50,
                lr            = 5e-4,
                device        = device,
                seed          = seed,
                pred_len      = pred_len,
                verbose       = False
            )
            print(f"Training time: {time.perf_counter() - t0:.1f}s")

            # ── Load best checkpoint and evaluate on test set ──────────────────
            ckpt = torch.load(
                f'best_autoformer_seed{seed}_pred{pred_len}.pth',
                map_location=device, weights_only=False
            )
            model.load_state_dict(ckpt['model_state_dict'])
            res = evaluate_model(model, test_loader, device)

            print(f"Test — MAE: {res['mae']:.2f}°C | RMSE: {res['rmse']:.2f}°C | "
                  f"R²: {res['r2_score']:.4f} | sMAPE: {res['smape_percent']:.2f}%")

            row = {
                'pred_len': pred_len, 'seed': seed,
                'mae':  res['mae'],    'rmse': res['rmse'],
                'r2':   res['r2_score'], 'smape': res['smape_percent']
            }
            horizon_results.append(row)
            all_results.append(row)

        # ── Per-horizon summary table ──────────────────────────────────────────
        hr = pd.DataFrame(horizon_results)
        print(f"\n{'='*62}")
        print(f"Summary — pred_len={pred_len}d  ({len(seeds)} seeds)")
        print(f"{'Metric':<14} {'Mean ± Std':<28} {'Best':<14} {'Worst'}")
        print("-"*62)
        for col, label, best_fn, worst_fn in [
            ('mae',   'MAE (°C)',   'min', 'max'),
            ('rmse',  'RMSE (°C)',  'min', 'max'),
            ('r2',    'R²',         'max', 'min'),
            ('smape', 'sMAPE (%)',  'min', 'max'),
        ]:
            best  = getattr(hr[col], best_fn)()
            worst = getattr(hr[col], worst_fn)()
            print(f"{label:<14} {hr[col].mean():.4f} ± {hr[col].std():.4f}"
                  f"              {best:.4f}   {worst:.4f}")

    # ── Save full results CSV ──────────────────────────────────────────────────
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('FAN-Autoformer_results_final.csv', index=False)
    print("\n📁 Full results saved to FAN-Autoformer_results_final.csv")

    # ── Final comparison table ────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("FAN-Autoformer — Final Results Table (mean ± std across all seeds)")
    print(f"{'pred_len':<10} {'MAE (°C)':<22} {'RMSE (°C)':<22} {'R²':<20} {'sMAPE (%)'}")
    print("-" * 72)
    for ph in PRED_HORIZONS:
        s = results_df[results_df['pred_len'] == ph]
        print(f"{ph:<10} "
              f"{s['mae'].mean():.2f} ± {s['mae'].std():.2f}           "
              f"{s['rmse'].mean():.2f} ± {s['rmse'].std():.2f}          "
              f"{s['r2'].mean():.4f} ± {s['r2'].std():.4f}  "
              f"{s['smape'].mean():.2f} ± {s['smape'].std():.2f}")
